# 01 - Data EDA
Exploratory analysis of `patient_records.csv` (Week 1). Run from the notebook's folder or the repo root.

> Student engineering project - not clinical guidance.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt, seaborn as sns, pandas as pd
from src import config
from src.data_pipeline import load_raw, load_and_clean, profile
raw = load_raw(ROOT / 'data' / 'patient_records.csv')
print(raw.shape); print(raw.dtypes); print(raw.isnull().sum())
raw.describe(include='all')

**Gotcha:** `pd.read_csv` treats the string `None` as missing. Here `None` is a real category (no prior reaction), so we load with `keep_default_na=False` (done inside `load_raw`).

In [ ]:
clean = load_and_clean(ROOT / 'data' / 'patient_records.csv')
print('missing after cleaning:', int(clean.isnull().sum().sum()))
print(clean[config.CATEGORICAL_COL].value_counts())

### Visualization 1 - distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, config.NUMERIC_COLS):
    sns.histplot(data=clean, x=col, hue=config.TARGET_COL, kde=True, ax=ax, palette='viridis')
    ax.set_title(f'{col} by Reaction_Score')
plt.tight_layout(); plt.show()

### Visualization 2 - class balance and prior-reaction effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
clean[config.TARGET_COL].value_counts().sort_index().plot.bar(ax=axes[0], color=['#0F6E6E', '#C0504D'])
axes[0].set_title('Class balance (0 = low, 1 = high risk)')
clean.groupby(config.CATEGORICAL_COL)[config.TARGET_COL].mean().reindex(config.CATEGORY_ORDER).plot.bar(ax=axes[1], color='#1F3A5F')
axes[1].set_title('High-risk rate by prior reaction history')
plt.tight_layout(); plt.show()

### Visualization 3 - correlation heatmap

In [ ]:
enc = clean.copy()
enc[config.CATEGORICAL_COL] = enc[config.CATEGORICAL_COL].map({c: i for i, c in enumerate(config.CATEGORY_ORDER)})
sns.heatmap(enc.drop(columns=[config.ID_COL]).corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlation matrix'); plt.show()

### Model comparison (3-5 sentences)
See `outputs/model_metrics.json`. Logistic regression beat the random forest on held-out macro F1 (about 0.72 vs 0.65). The signal in this dataset is noisy and mostly additive (dose, biomarker, age, conditions), which suits a regularised linear model; with only 400 training rows the 200-tree forest overfits that noise. The saved model is therefore the logistic-regression pipeline, selected automatically by test F1.